# 07 · Chunk, embed and index

> **Run order.** This notebook is step 7 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Elements become chunks, chunks become vectors, vectors go into Qdrant.

**Three chunking rules, each with a reason:**

1. A table is never separated from its header. Rows are split when they must be,
   but every slice repeats the header — `12,345` with no column label is noise.
2. Text is grouped under the heading above it. "Revenue grew 12%" is ambiguous
   without "Speciality segment" over it, and an ambiguous sentence embeds to an
   ambiguous vector.
3. Every chunk keeps the element IDs it came from. Retrieval returns a chunk; a
   *citation* must point at an element on a page.

Qdrant holds vectors and a pointer, **never an authoritative value** — a hit
gives back `element_ids` and the number is read from Postgres. See
[ADR-005](../docs/adr/0005-vector-store.md).

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from sqlalchemy import select
from analyst.chunking import MAX_TABLE_CHARS, TARGET_CHARS, SourceElement, chunk_document
from analyst.db import session_scope
from analyst.models import Document, ElementRow

chunks = []
with session_scope() as s:
    docs = [(d.document_id, d.ticker, d.fiscal_year)
            for d in s.execute(select(Document).order_by(Document.ticker)).scalars().all()]
    for document_id, ticker, fy in docs:
        rows = s.execute(
            select(ElementRow.element_id, ElementRow.document_id, ElementRow.page,
                   ElementRow.seq, ElementRow.type, ElementRow.text, ElementRow.table_json)
            .where(ElementRow.document_id == document_id)
            .order_by(ElementRow.page, ElementRow.seq)).all()
        els = [SourceElement(element_id=r[0], document_id=r[1], page=r[2], seq=r[3],
                             type=r[4], text=r[5], table_json=r[6]) for r in rows]
        chunks.extend(chunk_document(els, ticker, fy))

df = pd.DataFrame([{"type": c.type, "chars": len(c.text), "ticker": c.ticker} for c in chunks])
print(f"TOTAL CHUNKS: {len(chunks):,}   (target {TARGET_CHARS} chars text, "
      f"{MAX_TABLE_CHARS} chars table)\n")
# describe() defaults to 25/50/75; ask for the 90th explicitly.
df.groupby("type")["chars"].describe(percentiles=[0.5, 0.9])[
    ["count", "mean", "50%", "90%", "max"]
].round(0)

### Why the table budget is separate and tighter

Dense numeric text tokenizes far worse than prose — `520,412.5` is several tokens, not one — so a table runs closer to 2.5 chars/token against ~4 for English. Before this cap existed the largest table chunk was **6,643 characters**, well past the encoder's 512-token limit, so most of it was silently truncated and never embedded. Content past the limit is not extra context; it is content the retriever cannot see.

## Embedding throughput — measure before optimising

Two things were measured on this machine, and one of them is counter-intuitive.

In [ ]:
import time
from fastembed import TextEmbedding

sample = ["Particulars | FY2025 | FY2024\n" +
          "Revenue from operations | 520,412.5 | 438,860.1\n" * 12] * 128

results = []
for label, init_kw, embed_kw in [
    ("default",     {},              {}),
    ("threads=16",  {"threads": 16}, {}),
    ("parallel=8",  {},              {"parallel": 8}),
]:
    m = TextEmbedding(model_name="BAAI/bge-small-en-v1.5", **init_kw)
    t0 = time.perf_counter()
    list(m.embed(sample, batch_size=64, **embed_kw))
    dt = time.perf_counter() - t0
    results.append({"config": label, "chunks_per_sec": round(len(sample) / dt, 1),
                    "seconds": round(dt, 1)})
pd.DataFrame(results)

**Read that carefully.** onnxruntime intra-op threads do nothing — a single model instance already saturates what it can use. Process-level data parallelism roughly doubles throughput, because the model is small enough that more *copies* beat more threads per copy.

## Index

In [ ]:
import time
from analyst.config import get_settings
from analyst.embedding import DEFAULT_MODEL, Embedder
from analyst.vectorstore import VectorStore

MODEL = DEFAULT_MODEL
BATCH = 256

settings = get_settings()
embedder = Embedder(MODEL)
store = VectorStore(settings.qdrant_url, f"{settings.collection_prefix}_{MODEL}", embedder.dim)
store.recreate()

t0 = time.perf_counter()
for i in range(0, len(chunks), BATCH):
    window = chunks[i : i + BATCH]
    store.upsert(window, list(embedder.embed_documents([c.text for c in window])))

elapsed = time.perf_counter() - t0
print(f"model      : {MODEL} ({embedder.dim}d, cosine)")
print(f"collection : {store.collection}")
print(f"points     : {store.count():,}")
print(f"elapsed    : {elapsed/60:.1f} min  ({len(chunks)/elapsed:.1f} chunks/sec)")

## Sanity check: does it retrieve anything sensible?

In [ ]:
q = "What was Sun Pharma's total revenue in FY2025?"
hits = store.search(embedder.embed_query(q), limit=5, ticker="SUNPHARMA")
pd.DataFrame([
    {"score": round(h.score, 3), "page": h.pages[0], "type": h.type,
     "text": h.text[:90].replace("\n", " ")}
    for h in hits
])